# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lthendral10/flyrank-ml-internship/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [8]:
!git clone https://github.com/lthendral10/flyrank-ml-internship.git

fatal: destination path 'flyrank-ml-internship' already exists and is not an empty directory.


## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


# Ranked Actions + Reason Codes

This action playbook ranks content pages based on observed SEO and engagement signals. Each recommendation includes a reason code to explain why the page appears in the queue.

Reason Codes:

- RC01 – Low CTR and high impressions
- RC02 – Old content with no recent update
- RC03 – Declining trend in performance
- RC04 – Low engagement rate
- RC05 – High search volume opportunity

Action Labels:

- Refresh Content
- Improve Metadata
- Update Keywords
- Improve User Engagement
- Monitor Performance

In [10]:
import pandas as pd
import numpy as np
df = pd.read_csv("/content/flyrank-ml-internship/data/raw/content_refresh_anonymized.csv")
# Create a simple priority score
df["priority_score"] = (
    df["days_since_last_update"].fillna(0) * 0.4 +
    df["search_volume"].fillna(0) * 0.3 +
    (100 - df["ctr"].fillna(0)) * 0.2 +
    df["content_age_days"].fillna(0) * 0.1
)

def get_reason(row):
    if row["ctr"] < 1:
        return "RC01"
    elif row["days_since_last_update"] > 365:
        return "RC02"
    elif row["trend_direction"] == "down":
        return "RC03"
    elif row["engagement_rate"] < 20:
        return "RC04"
    else:
        return "RC05"

df["reason_code"] = df.apply(get_reason, axis=1)

reason_to_action = {
    "RC01": "Improve Metadata",
    "RC02": "Refresh Content",
    "RC03": "Review Content",
    "RC04": "Improve User Engagement",
    "RC05": "Monitor Performance"
}

df["action"] = df["reason_code"].map(reason_to_action)

queue = df.sort_values("priority_score", ascending=False)

queue[[
    "content_id",
    "priority_score",
    "reason_code",
    "action"
]].head(20)

,content_id,priority_score,reason_code,action
12140,content_ef99c4abd9ab,22307.894,RC01,Improve Metadata
28282,content_454cc6654c6e,18257.900,RC01,Improve Metadata
17907,content_5ec29ae79c60,18257.900,RC01,Improve Metadata
6972,content_bf67a444faef,18257.900,RC01,Improve Metadata
18701,content_deb54e9e19cd,18257.900,RC01,Improve Metadata
8055,content_cd6760921db8,14932.700,RC01,Improve Metadata
16005,content_83e3da1394ac,14925.100,RC01,Improve Metadata
22788,content_ee4630879d03,14924.300,RC01,Improve Metadata
13502,content_f76ccf7a7834,14923.270,RC01,Improve Metadata
15923,content_84fe9d0a707a,12257.900,RC01,Improve Metadata


## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

In [11]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


# Intended Use and Limits

This action playbook is designed to support SEO analysts and content teams in prioritizing pages for review.

The recommendations are based on observed search and engagement signals and should be treated as decision-support rather than automatic decisions.

The recommendations should always be reviewed by a human before implementation.

In [12]:
print("Number of content pages:", len(df))

print("\nAction Distribution")
print(queue["action"].value_counts())

Number of content pages: 30000

Action Distribution
action
Improve Metadata           28291
Improve User Engagement      880
Review Content               753
Monitor Performance           75
Refresh Content                1
Name: count, dtype: int64


## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

In [13]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


# Human Review

Before taking action, a reviewer should check:

- Current content quality
- Search intent
- Keyword relevance
- Business priorities
- Recent manual updates

The following should never be automated:

- Publishing content changes
- Deleting content
- Redirecting pages
- Changing website structure
- Making business decisions without review

In [14]:
review_sample = queue[[
    "content_id",
    "reason_code",
    "action",
    "priority_score"
]].head(10)

review_sample

,content_id,reason_code,action,priority_score
12140,content_ef99c4abd9ab,RC01,Improve Metadata,22307.894
28282,content_454cc6654c6e,RC01,Improve Metadata,18257.900
17907,content_5ec29ae79c60,RC01,Improve Metadata,18257.900
6972,content_bf67a444faef,RC01,Improve Metadata,18257.900
18701,content_deb54e9e19cd,RC01,Improve Metadata,18257.900
8055,content_cd6760921db8,RC01,Improve Metadata,14932.700
16005,content_83e3da1394ac,RC01,Improve Metadata,14925.100
22788,content_ee4630879d03,RC01,Improve Metadata,14924.300
13502,content_f76ccf7a7834,RC01,Improve Metadata,14923.270
15923,content_84fe9d0a707a,RC01,Improve Metadata,12257.900


## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

In [15]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


# Monitoring and Retraining

The recommendations should be reviewed regularly.

Possible retraining triggers include:

- Significant changes in search performance
- New content added
- Major search algorithm updates
- Changes in user engagement
- Performance degradation of the model

These observations are intended for decision-support and should be monitored over time.

In [16]:
monitoring = pd.DataFrame({
    "Trigger": [
        "CTR decreases",
        "Engagement decreases",
        "Search volume changes",
        "Content becomes older",
        "Model performance drops"
    ],
    "Suggested Action": [
        "Review metadata",
        "Improve content",
        "Recalculate scores",
        "Refresh content",
        "Retrain model"
    ]
})

monitoring

,Trigger,Suggested Action
0,CTR decreases,Review metadata
1,Engagement decreases,Improve content
2,Search volume changes,Recalculate scores
3,Content becomes older,Refresh content
4,Model performance drops,Retrain model


## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

In [17]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


# Export Results

The ranked action queue is exported so it can be reused in the final research paper.

In [18]:
import os

# Create output folder
os.makedirs("work/outputs", exist_ok=True)

# Export ranked queue
queue.to_csv(
    "work/outputs/content_action_queue.csv",
    index=False
)

print("Queue exported successfully.")

# Export top 20 only (optional)
queue.head(20).to_csv(
    "work/outputs/top20_action_queue.csv",
    index=False
)

print("Top 20 queue exported.")

Queue exported successfully.
Top 20 queue exported.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.